# Inventory Review & Approval

## Overview

Review the generated inventory and copy approved dashboards to the QC-approved location.

**What this notebook does:**
1. Loads inventory.csv from dashboard_inventory/
2. Displays summary statistics and full data
3. Allows you to filter/review dashboards
4. Copies approved inventory to dashboard_inventory_approved/

**Output:** `dashboard_inventory_approved/inventory.csv` ready for Step 2

---

## Cell 1: Configuration

Update these paths to match your environment

In [ ]:
# Volume paths (update to match your environment)
VOLUME_BASE = "/Volumes/archana_krish_fe_dsa/vizient_deep_dive/dashboard_migration"
INVENTORY_SOURCE = f"{VOLUME_BASE}/dashboard_inventory/inventory.csv"
INVENTORY_APPROVED = f"{VOLUME_BASE}/dashboard_inventory_approved/inventory.csv"

print(f"📁 Source inventory: {INVENTORY_SOURCE}")
print(f"✅ Approved inventory: {INVENTORY_APPROVED}")

## Cell 2: Load Inventory

In [ ]:
# Load inventory CSV
print("📥 Loading inventory...\n")

df = spark.read.csv(INVENTORY_SOURCE, header=True, inferSchema=True)

print(f"✅ Loaded {df.count()} dashboards\n")
print("="*70)
print("INVENTORY SUMMARY")
print("="*70)

# Summary statistics
print(f"\n📊 Total Dashboards: {df.count()}")
print(f"\n🎯 Activity Level Distribution:")
df.groupBy('activity_level').count().orderBy('count', ascending=False).show()

print(f"\n🔧 Complexity Distribution:")
df.groupBy('complexity').count().orderBy('count', ascending=False).show()

print(f"\n📈 Table Usage Stats:")
df.select('table_count').summary('min', 'max', 'mean').show()

print("\n" + "="*70)

## Cell 3: View Full Inventory

Review all dashboards - scroll through to identify any issues

In [ ]:
# Display full inventory for review
print("📋 Full Inventory (sorted by table count):\n")
display(df.orderBy('table_count', ascending=False))

## Cell 4: Identify Issues

Check for dashboards with potential problems

In [ ]:
# Check for dashboards with fallback names (API fetch failed)
print("⚠️  Dashboards with Fallback Names (API fetch failed):\n")
fallback_names = df.filter(df.dashboard_name.startswith('Dashboard_'))
fallback_count = fallback_names.count()

if fallback_count > 0:
    print(f"Found {fallback_count} dashboards with fallback names:")
    display(fallback_names.select('dashboard_id', 'dashboard_name', 'table_count', 'activity_level'))
    print("\n💡 These dashboards likely no longer exist or are inaccessible.")
    print("   Recommendation: EXCLUDE from migration\n")
else:
    print("✅ All dashboards have real names\n")

# Check for inactive dashboards
print("\n📊 Inactive Dashboards (no recent usage):\n")
inactive = df.filter(df.activity_level == 'Inactive')
inactive_count = inactive.count()

if inactive_count > 0:
    print(f"Found {inactive_count} inactive dashboards:")
    display(inactive.select('dashboard_id', 'dashboard_name', 'table_count', 'last_accessed'))
    print("\n💡 Consider excluding these from migration if unused\n")
else:
    print("✅ All dashboards have recent activity\n")

# Check for dashboards with no table references
print("\n📊 Dashboards with Zero Tables:\n")
no_tables = df.filter(df.table_count == 0)
no_tables_count = no_tables.count()

if no_tables_count > 0:
    print(f"Found {no_tables_count} dashboards with no table references:")
    display(no_tables.select('dashboard_id', 'dashboard_name', 'activity_level'))
    print("\n💡 These may be text-only dashboards or have issues\n")
else:
    print("✅ All dashboards reference tables\n")

## Cell 5: Filter & Approve

**CUSTOMIZE THIS CELL** with your filtering criteria

In [ ]:
# ============================================================================
# FILTERING LOGIC - Customize as needed
# ============================================================================

print("🎯 Applying QC filters...\n")

# Start with full inventory
df_approved = df

# Filter 1: Remove dashboards with fallback names (failed API lookups)
print("Filter 1: Removing dashboards with fallback names...")
df_approved = df_approved.filter(~df_approved.dashboard_name.startswith('Dashboard_'))
print(f"  Remaining: {df_approved.count()}\n")

# Filter 2: Only include dashboards with table references
print("Filter 2: Requiring table_count > 0...")
df_approved = df_approved.filter(df_approved.table_count > 0)
print(f"  Remaining: {df_approved.count()}\n")

# Filter 3: Remove inactive dashboards (optional - uncomment to enable)
# print("Filter 3: Removing inactive dashboards...")
# df_approved = df_approved.filter(df_approved.activity_level != 'Inactive')
# print(f"  Remaining: {df_approved.count()}\n")

# Filter 4: Custom filter (example - uncomment and modify as needed)
# print("Filter 4: Custom filter...")
# df_approved = df_approved.filter(df_approved.unique_users > 1)  # At least 2 users
# print(f"  Remaining: {df_approved.count()}\n")

print("="*70)
print(f"✅ APPROVED: {df_approved.count()} dashboards")
print(f"❌ EXCLUDED: {df.count() - df_approved.count()} dashboards")
print("="*70)

## Cell 6: Review Approved Inventory

In [ ]:
# Display approved inventory for final review
print("📋 APPROVED INVENTORY FOR MIGRATION:\n")
display(df_approved.orderBy('table_count', ascending=False))

# Summary of approved dashboards
print("\n📊 Approved Dashboard Summary:")
print(f"  Total: {df_approved.count()}")
print(f"  Avg tables/dashboard: {df_approved.agg({'table_count': 'avg'}).collect()[0][0]:.1f}")
print(f"  Total unique tables: {df_approved.agg({'unique_tables': 'sum'}).collect()[0][0]}")
print(f"\n  Activity Levels:")
df_approved.groupBy('activity_level').count().orderBy('count', ascending=False).show(truncate=False)
print(f"\n  Complexity Levels:")
df_approved.groupBy('complexity').count().orderBy('count', ascending=False).show(truncate=False)

## Cell 7: Save to Approved Location

**⚠️ IMPORTANT**: Only run this cell after reviewing the approved inventory above!

In [ ]:
# Ensure approved directory exists
approved_dir = f"{VOLUME_BASE}/dashboard_inventory_approved"
dbutils.fs.mkdirs(approved_dir)

# Convert to pandas and save as CSV
print("💾 Saving approved inventory...\n")
approved_pandas = df_approved.toPandas()

# Write to volume using dbutils
csv_content = approved_pandas.to_csv(index=False)
dbutils.fs.put(INVENTORY_APPROVED, csv_content, overwrite=True)

print("="*70)
print("✅ APPROVED INVENTORY SAVED")
print("="*70)
print(f"📁 Location: {INVENTORY_APPROVED}")
print(f"📊 Dashboards: {len(approved_pandas)}")
print(f"💾 Size: {len(csv_content) / 1024:.1f} KB")
print("="*70)
print("\n🎯 Next Steps:")
print("   1. Update databricks.yml to use 'dashboard_inventory_approved'")
print("   2. Redeploy: databricks bundle deploy -t dev")
print("   3. Run Step 2: databricks bundle run export_transform -t dev")
print("="*70)